# ImageNet Training Loop

In this notebook, we will run an ImageNet training loop on a single GPU in AWS. Eventually we will create a .py file to train ImageNet on multiple GPUs in AWS or RunPod.

In [1]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../..'))
import time

In [2]:
import torch
import torchvision
from torch.optim import SGD
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.profiler import profile, ProfilerActivity, record_function, schedule

from utils.imagenet import get_train_transform, get_val_transform
from utils.metrics import accuracy, topk_accuracy

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(device, torch.cuda.device_count(), torch.cuda.get_device_name(0))

cuda 1 NVIDIA GeForce RTX 5090


# Hyperparameters

In [4]:
# Mount network volume
bucket_path = '/workspace/imagenet'

# path to save model
save_path = os.getcwd()

# single epoch for testing. Eventually will set to 90
num_epochs = 90

# may need to be smaller if OOM occurs
batch_size = 256

# Change depending on number of CPUs, optimize
num_workers = 48
pin_memory = True
persistent_workers=False

# How many batches before logging loss
log_every = 500

# How many epochs before checkpointing model
checkpoint_every = 2

# How many times to perform validation per epoch
val_per_epoch = 2

# Optimizer hyperparameters
opt_kwargs = {'lr': 0.1 * batch_size/256, 'momentum': 0.9}
num_warmup = 5
T_max = 90
eta_min = 1e-5

# Fixed for ImageNet Dataset
C = 3
H, W = 224, 224
num_classes = 1000

# Import dataset

In [5]:
t1 = time.time()
val_ds = torchvision.datasets.ImageFolder(bucket_path + "/val", transform=get_val_transform())
t2 = time.time()
print(f"Time to create validation dataset: {t2-t1}")

Time to create validation dataset: 1.295328140258789


In [6]:
t1 = time.time()
train_ds = torchvision.datasets.ImageFolder(bucket_path + "/train", transform=get_train_transform())
t2 = time.time()
print(f"Time to create training dataset: {t2-t1}")

Time to create training dataset: 5.719165563583374


# Create dataloader

In [7]:
train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, 
    num_workers=num_workers, pin_memory=pin_memory, drop_last=True,
    persistent_workers=persistent_workers,
)
print(f"Length of training dataloader is {len(train_dl)}")

Length of training dataloader is 5004


In [8]:
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=2*batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, drop_last=False,
    persistent_workers=persistent_workers,
)
print(f"Length of validation dataloader is {len(val_dl)}")

Length of validation dataloader is 98


# Function to checkpoint model

In [9]:
def save_checkpoint(model, optimizer, epoch, train_records, val_records, path=save_path):
    checkpoint = {
        'epochs': epoch+1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_records': torch.tensor(train_records),
        'val_records': torch.tensor(val_records)
    }
    torch.save(checkpoint, path + '/imagenet-checkpoint.pt')
    print(f"Checkpoint saved after {epoch+1} epochs")

# Introduce loss and training metrics

In [10]:
loss_fn = torch.nn.functional.cross_entropy
metric_fns = [loss_fn, accuracy, topk_accuracy]

# Write functions for training loop

In [11]:
def is_log_batch(i):
    return i % log_every == 0 and i > 0

def is_validate_batch(i):
    return (i * val_per_epoch) % len(train_dl) < val_per_epoch

def is_checkpoint_epoch(epoch):
    return epoch % checkpoint_every == 0 and epoch > 0

In [12]:
def compute_metrics(metric_fns, running_metrics, logits, y, reduction='mean'):
    for metric_fn, running_metric in zip(metric_fns, running_metrics):
        running_metric += metric_fn(logits, y, reduction=reduction)

def log_metrics(records, running_metrics, div=1):
    log(records, running_metrics, div=div)
    reset_metric(running_metrics)

def log(records, metrics, div=1):
    for record, metric in zip(records, metrics):
        record.append(metric.item() / div)

def reset_metric(running_metrics):
    for tensor in running_metrics:
        tensor.fill_(0.0)

In [13]:
def train_step(model, X, y, opt, scaler):
    model.train()
    with torch.autocast(device.type):
        logits = model(X)
        
    loss = loss_fn(logits, y)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    opt.zero_grad(set_to_none=True)
    return logits

In [14]:
def val_step(model, X):
    model.eval()
    with torch.autocast(device.type):
        logits = model(X)
    return logits

In [15]:
def validate(val_dl, model, metric_fns, val_records, running_val_metrics):
    with torch.no_grad():
        for j, (X, y) in enumerate(val_dl):
            X = X.to(device, non_blocking=pin_memory)
            y = y.to(device, non_blocking=pin_memory)
            logits = val_step(model, X)
            compute_metrics(metric_fns, running_val_metrics, logits, y, reduction='sum')
    log_metrics(val_records, running_val_metrics, div=len(val_ds))

# Define model, optimizer, learning rate scheduler, and automatic mixed-precision (AMP) scaler

In [16]:
def get_new_model():
    return torchvision.models.resnet50().to(device)

In [17]:
def get_optimizer(model):
    return SGD(model.parameters(), **opt_kwargs)

In [18]:
def get_scaler():
    return torch.amp.GradScaler(device.type)

In [19]:
def get_scheduler(opt):
    scheduler1 = LinearLR(opt, 0.01, 1.0, num_warmup)
    scheduler2 = CosineAnnealingLR(opt, T_max=T_max-num_warmup, eta_min=eta_min)
    return SequentialLR(opt, schedulers=[scheduler1, scheduler2], milestones=[num_warmup])

# Profile model

In [ ]:
def training_epoch_to_profile(model, opt, scaler):
    running_metrics = [torch.tensor(0.0, device=device) for _ in metric_fns]

    for i, (X, y) in enumerate(train_dl):
        X = X.to(device, non_blocking=pin_memory)
        y = y.to(device, non_blocking=pin_memory)

        if is_log_batch(i):
            print(f"Batch {i}")
    
        logits = train_step(model, X, y, opt, scaler)

        with torch.no_grad():
            compute_metrics(metric_fns, running_metrics, logits, y)

        if i > num_workers * 5:
            break

In [ ]:
def trace_handler(prof):
    print(prof.key_averages().table(sort_by="self_cpu_time_total", row_limit=10))

In [ ]:
activities = [ProfilerActivity.CPU, ProfilerActivity.CUDA]

with torch.profiler.profile(
    activities=activities,
    schedule=schedule(wait=1, warmup=1, active=1, skip_first=0, repeat=0),
    on_trace_ready=trace_handler
) as prof:
    model_prof = get_new_model()
    opt_prof = get_optimizer(model_prof)
    scaler_prof = get_scaler()
    for epoch in range(3):
        print(f"Epoch: {epoch}")
        training_epoch_to_profile(model_prof, opt_prof, scaler_prof)
        prof.step()

In [ ]:
prof.export_chrome_trace("trace.json")

# Create model, optimizer, scheduler, and AMP scaler

In [20]:
model = get_new_model()

In [21]:
opt = get_optimizer(model)

In [22]:
scheduler = get_scheduler(opt)

In [23]:
scaler = get_scaler()

# Run training loop

In [ ]:
train_records = [[] for _ in range(len(metric_fns))]
val_records = [[] for _ in range(len(metric_fns))]
running_train_metrics = [torch.tensor(0.0, device=device) for _ in metric_fns]
running_val_metrics = [torch.tensor(0.0, device=device) for _ in metric_fns]

for epoch in range(num_epochs):
    t1 = time.time()
    print(f"Epoch {epoch+1}/{num_epochs}")

    for i, (X, y) in enumerate(train_dl):
        X = X.to(device, non_blocking=pin_memory)
        y = y.to(device, non_blocking=pin_memory)

        if is_log_batch(i):
            log_metrics(train_records, running_train_metrics, div=log_every)
            print(f"Train loss & accuracy at batch {i}/{len(train_dl)}: "
                   f"{train_records[0][-1]:.5f}, {train_records[1][-1]:.5f}")
        
        logits = train_step(model, X, y, opt, scaler)

        with torch.no_grad():
            compute_metrics(metric_fns, running_train_metrics, logits, y)

        if is_validate_batch(i):
            validate(val_dl, model, metric_fns, val_records, running_val_metrics)
            print(f"Val loss & accuracy at batch {i}/{len(train_dl)}: "
                  f"{val_records[0][-1]:.5f}, {val_records[1][-1]:.5f}")

    scheduler.step()

    if is_checkpoint_epoch(epoch):
        save_checkpoint(model, opt, epoch, train_records, val_records)

    t2 = time.time()
    print(f"Time to run epoch {epoch}: {(t2-t1)/60:.2f} minutes")
        
# Calculate final validation metrics
validate(val_dl, model, metric_fns, val_records, running_val_metrics)

Epoch 1/90


In [ ]:
# finally, save the model and metrics
save_checkpoint(model, opt, epoch, train_records, val_records)